# 1-캡스톤. Streamlit 기반 금융 규정 Q&A 에이전트

Day 1에서 배운 **LLM · 프롬프트 · Tool Calling · RAG · LangGraph Agent · MemorySaver**를 하나의 웹 앱으로 엮는 캡스톤 노트북이다. 노트북 안에서 `%%writefile` 매직으로 `app.py`를 생성한 뒤, 터미널에서 `streamlit run app.py`로 띄워 쓴다.

## 학습 목표
- 앞선 세션의 각 부품이 **실제 제품** 안에서 어떻게 조립되는지 체감한다.
- `create_react_agent` + `MemorySaver` + Chroma 리트리버를 **Streamlit 세션 모델**과 연결한다.
- `agent.stream(stream_mode='updates')` 이벤트를 해석해 **도구 호출 과정을 UI에 노출**한다.

## 복습 매핑
| 부품 | 유래 | 이 캡스톤에서의 역할 |
|---|---|---|
| `get_chat_model` / `get_embeddings` | 1-5 LLM & Inference | LLM / 임베딩 팩토리 |
| Chroma + MMR retriever | 2-3 Retriever Patterns · 2-4 RAG Pipeline | `search_regulation` 도구의 몸통 |
| `@tool`, ReAct, `create_react_agent` | 2-6 Agent & LangGraph | 도구 루프 |
| `MemorySaver`, `thread_id` | 2-6 Agent & LangGraph | 대화 기억 |

## 핵심 키워드
`Streamlit` · `ReAct Agent` · `MemorySaver` · `thread_id` · `Chroma` · `DuckDuckGo` · `%%writefile`

## 0. 사전 요구사항

1. **Chroma 인덱스**: `day1/session2_rag_basics/04_rag_pipeline_lcel.ipynb`를 한 번 실행해 `_store/efinance_rag` (컬렉션명 `efinance_regulation`)가 만들어져 있어야 한다.
2. **의존성**: `streamlit`, `ddgs`는 프로젝트 `pyproject.toml`의 의존성으로 등록되어 있다. `uv sync`만 하면 자동으로 설치된다.
   > ℹ️ `langchain_community.tools.DuckDuckGoSearchRun`은 내부적으로 리브랜딩된 `ddgs` 패키지를 import한다 (구버전 `duckduckgo-search`가 아님).
   > 혹시 `uv`를 쓰지 않는 환경이라면 아래 다음 셀의 `%pip install`로 개별 설치도 가능.
3. **`.env`**: 앞선 노트북과 동일하게 `OPENAI_API_KEY` (+ 선택적으로 `OPENAI_API_BASE`, `OPENAI_MODEL`)가 프로젝트 루트의 `.env`에 설정되어 있어야 한다.

In [ ]:
# uv sync 한 환경이면 이 셀은 no-op. 순수 pip 환경이면 아래로 개별 설치.
# %pip install -q streamlit ddgs

In [5]:
import sys; sys.path.insert(0, '../..')
from pathlib import Path

CHROMA_DIR = Path('../session2_rag_basics/_store/efinance_rag')
assert CHROMA_DIR.exists(), (
    f'Chroma 인덱스가 없습니다: {CHROMA_DIR.resolve()}\n'
    '먼저 day1/session2_rag_basics/04_rag_pipeline_lcel.ipynb 를 실행해 인덱스를 만들어주세요.'
)

from common import provider_badge
print(provider_badge())
print(f'✅ Chroma 인덱스 확인: {CHROMA_DIR.resolve()}')

☁️ OpenAI | model=openai/gpt-4o-mini
✅ Chroma 인덱스 확인: /home/bong/dev/airgap-llm-edu/day1/session2_rag_basics/_store/efinance_rag


## 1. Streamlit 30초 정리

| 개념 | 동작 |
|---|---|
| **스크립트 재실행 모델** | 사용자의 모든 상호작용(버튼 클릭, 입력 제출)마다 `app.py` 전체가 **위에서 아래로 다시 실행**된다. |
| `st.session_state` | 재실행에도 살아남는 **세션 단위 메모리**. 대화 기록·`thread_id` 같은 상태를 여기 보관한다. |
| `@st.cache_resource` | LLM·리트리버처럼 **프로세스 생애에 1번만** 만들고 싶은 리소스를 캐시. 재실행마다 새로 만들지 않는다. |
| `st.chat_input` / `st.chat_message` | 채팅 UI 기본 블록. `st.chat_input`은 페이지 하단 고정 입력창, `st.chat_message`는 role 기반 말풍선. |
| `st.empty()` / `st.container()` | 스트리밍 중 **업데이트되는 자리**를 미리 확보해두는 플레이스홀더. |

> 💡 스트림릿의 "매번 재실행" 모델 때문에 **LangGraph의 `MemorySaver`는 `@st.cache_resource`로 감싸 싱글톤으로 유지**해야 한다. 그렇지 않으면 매 입력마다 새 `MemorySaver`가 만들어져 대화가 유실된다.

## 2. 에이전트 구조

```
                    ┌──────────────────────────────┐
                    │  Streamlit UI (app.py)      │
                    │  - st.chat_input            │
                    │  - thread_id in session     │
                    │  - tool_call expanders      │
                    └──────────────┬───────────────┘
                                   │ agent.stream(config={thread_id})
                                   ▼
              ┌─────────────────────────────────────┐
              │  LangGraph ReAct Agent              │
              │  (create_react_agent)               │
              │  + MemorySaver (checkpointer)       │
              └──┬──────────┬─────────────┬─────────┘
                 │          │             │
                 ▼          ▼             ▼
         ┌───────────┐ ┌───────────┐ ┌───────────┐
         │ search_   │ │ calculate │ │ web_search│
         │ regulation│ │ _interest │ │ (DDG)     │
         └────┬──────┘ └───────────┘ └───────────┘
              │
              ▼
     Chroma(efinance_rag) + MMR
```

### 도구 3종

| 도구 | 무엇을 하나 | 비고 |
|---|---|---|
| `search_regulation(query)` | **기존 Chroma `efinance_rag`** 를 MMR로 검색해 조항 본문 반환 | 2-4 노트북에서 만든 인덱스 **그대로 재사용** (읽기 전용) |
| `calculate_interest(principal, annual_rate_pct, days)` | 단리 이자 계산 | LLM이 수치를 직접 추론하지 않도록 강제하는 **가드 도구** |
| `web_search(query)` | DuckDuckGo로 웹 상위 결과 요약 | 폐쇄망 배포 시 **제거하거나 사내 검색 API로 교체** 필요 |

### 스트리밍 이벤트 해석 (`stream_mode='updates'`)

`agent.stream`은 **노드 완료 단위**로 다음과 같은 dict를 내보낸다:

```python
{'agent': {'messages': [AIMessage(content='...', tool_calls=[{'name': ..., 'args': ...}])]}}  # LLM 턴
{'tools': {'messages': [ToolMessage(name='search_regulation', content='...')]}}                # 도구 실행 결과
```

UI 파싱 규칙:
- `AIMessage.tool_calls`가 있으면 → "🔧 도구 호출" expander로 인자 표시
- `ToolMessage`가 오면 → "👁 도구 결과" expander로 내용 표시
- `AIMessage.content`만 있으면 → 최종 답변으로 말풍선에 렌더

## 3. `app.py` 생성

아래 셀을 실행하면 **현재 노트북과 같은 디렉터리**(`day1/session3_capstone/`)에 `app.py`가 만들어진다. `%%writefile`은 셀 내용을 해당 파일에 그대로 저장하는 IPython 매직이다.

In [6]:
%%writefile app.py
"""Day1 캡스톤: 금융 규정 Q&A 에이전트 (Streamlit + LangGraph ReAct + MemorySaver).

실행:
    cd day1/session3_capstone
    streamlit run app.py

사전 준비:
    day1/session2_rag_basics/04_rag_pipeline_lcel.ipynb 를 한 번은 실행해
    _store/efinance_rag (collection='efinance_regulation') 가 만들어져 있어야 합니다.
"""
from __future__ import annotations

import sys
import uuid
from pathlib import Path

import streamlit as st

# 프로젝트 루트를 import path에 추가 → `common` 모듈 로드
ROOT = Path(__file__).resolve().parent.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common import get_chat_model, get_embeddings, provider_badge  # noqa: E402

from langchain_core.tools import tool  # noqa: E402
from langchain_community.vectorstores import Chroma  # noqa: E402
from langchain_community.tools import DuckDuckGoSearchRun  # noqa: E402
from langgraph.prebuilt import create_react_agent  # noqa: E402
from langgraph.checkpoint.memory import MemorySaver  # noqa: E402

CHROMA_DIR = ROOT / 'day1' / 'session2_rag_basics' / '_store' / 'efinance_rag'
COLLECTION = 'efinance_regulation'

SYSTEM_PROMPT = """당신은 금융회사의 내부 규정 Q&A를 돕는 AI 어시스턴트입니다.

도구 사용 규칙:
1. 전자금융감독규정 관련 질문은 반드시 `search_regulation` 도구를 **먼저** 호출해 근거를 확보하세요.
2. 이자·기간·금액 계산이 필요하면 반드시 `calculate_interest`를 사용하세요. 직접 계산하지 마세요.
3. 규정 밖의 최신 정보(예: 최근 공시, 뉴스)는 `web_search`로 찾되, 출처 URL을 같이 제시하세요.
4. 도구로도 확인할 수 없는 질문에는 \"확인할 수 없습니다\"라고 답하세요.

답변 형식:
- 한국어로 간결하게.
- 규정 근거는 `[근거: 제N조]`, 웹 출처는 `[출처: URL]` 형식으로 표기.
"""


# ──────────────── 리소스 로더 (프로세스 당 1회) ────────────────
@st.cache_resource(show_spinner='🔌 Chroma 인덱스 로드 중...')
def load_retriever():
    if not CHROMA_DIR.exists():
        st.error(
            f'Chroma 인덱스가 없습니다: {CHROMA_DIR}\n\n'
            '먼저 `day1/session2_rag_basics/04_rag_pipeline_lcel.ipynb` 를 실행해주세요.'
        )
        st.stop()
    vs = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=get_embeddings(),
        collection_name=COLLECTION,
    )
    return vs.as_retriever(
        search_type='mmr',
        search_kwargs={'k': 4, 'fetch_k': 12, 'lambda_mult': 0.5},
    )


@st.cache_resource(show_spinner='🛠 에이전트 구성 중...')
def build_agent():
    retriever = load_retriever()
    ddg = DuckDuckGoSearchRun()

    @tool
    def search_regulation(query: str) -> str:
        """전자금융감독규정을 벡터 검색해 관련 조항 본문을 반환한다.
        반환된 '[제N조(...)]' 태그는 최종 답변에서 '[근거: 제N조]' 형식으로 인용할 것.

        Args:
            query: 검색어 (자연어 질의 가능 — 예: '해킹 방지대책의 주요 통제항목')
        """
        docs = retriever.invoke(query)
        if not docs:
            return '관련 규정을 찾지 못했습니다.'
        return '\n\n'.join(
            f'[{d.metadata.get("article", "알 수 없음")}] {d.page_content}'
            for d in docs
        )

    @tool
    def calculate_interest(principal: float, annual_rate_pct: float, days: int) -> float:
        """원금, 연이자율(%), 기간(일)으로 단리 이자를 계산한다.

        Args:
            principal: 원금 (원)
            annual_rate_pct: 연 이자율 (%)
            days: 예치 기간 (일)
        """
        return round(principal * annual_rate_pct / 100 * days / 365, 2)

    @tool
    def web_search(query: str) -> str:
        """DuckDuckGo로 웹을 검색해 상위 결과의 요약을 반환한다.
        규정/도메인 지식으로 답할 수 없는 최신 정보(뉴스·공시 등)에만 사용.
        폐쇄망 배포 시 이 도구는 **제거하거나 사내 검색 API로 교체**해야 한다.

        Args:
            query: 검색어
        """
        try:
            return ddg.invoke(query)
        except Exception as e:
            return f'웹 검색 실패: {type(e).__name__}: {e}'

    tools = [search_regulation, calculate_interest, web_search]
    llm = get_chat_model(temperature=0)
    memory = MemorySaver()
    return create_react_agent(llm, tools, checkpointer=memory, prompt=SYSTEM_PROMPT)


# ──────────────── Streamlit UI ────────────────
st.set_page_config(page_title='금융 규정 Q&A 에이전트', page_icon='🏦', layout='wide')
st.title('🏦 금융 규정 Q&A 에이전트')
st.caption(f'{provider_badge()}  ·  ReAct + MemorySaver + Chroma RAG')

# 세션 상태 초기화
if 'thread_id' not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())
if 'messages' not in st.session_state:
    st.session_state.messages = []  # [{role, content, tool_events}]

# ── 사이드바 ──
with st.sidebar:
    st.subheader('🧵 대화 세션')
    st.caption('thread_id (MemorySaver 키)')
    st.code(st.session_state.thread_id, language=None)
    if st.button('🆕 새 대화', use_container_width=True):
        st.session_state.thread_id = str(uuid.uuid4())
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.subheader('🛠 도구')
    st.markdown(
        '- `search_regulation` — 전자금융감독규정 Chroma RAG\n'
        '- `calculate_interest` — 단리 이자 계산\n'
        '- `web_search` — DuckDuckGo 웹 검색'
    )

    st.divider()
    st.subheader('💡 예시 질문')
    st.markdown(
        '- 해킹 등 방지대책의 주요 통제항목을 규정에서 찾아줘\n'
        '- 1000만원을 연 3.5%로 90일 예치하면 이자가 얼마?\n'
        '- 전자금융거래법 관련 최근 금감원 제재 뉴스 찾아줘'
    )


# ── 렌더 헬퍼 ──
def render_tool_events(events: list[dict]) -> None:
    for ev in events:
        if ev['kind'] == 'call':
            with st.expander(f"🔧 `{ev['name']}` 호출", expanded=False):
                st.json(ev['args'])
        elif ev['kind'] == 'result':
            with st.expander(f"👁 `{ev['name']}` 결과", expanded=False):
                st.code(ev['content'][:2000])


# ── 과거 메시지 렌더 ──
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        if msg['role'] == 'assistant':
            render_tool_events(msg.get('tool_events', []))
        st.markdown(msg['content'])


# ── 입력 처리 ──
if user_input := st.chat_input('규정·이자 계산·최근 뉴스 등을 물어보세요'):
    st.session_state.messages.append({
        'role': 'user',
        'content': user_input,
        'tool_events': [],
    })
    with st.chat_message('user'):
        st.markdown(user_input)

    agent = build_agent()
    config = {'configurable': {'thread_id': st.session_state.thread_id}}

    with st.chat_message('assistant'):
        tool_slot = st.container()
        answer_slot = st.empty()

        tool_events: list[dict] = []
        final_answer = ''

        try:
            for event in agent.stream(
                {'messages': [('user', user_input)]},
                config=config,
                stream_mode='updates',
            ):
                for _node, payload in event.items():
                    for m in payload.get('messages', []):
                        tool_calls = getattr(m, 'tool_calls', None)
                        if tool_calls:
                            for c in tool_calls:
                                ev = {'kind': 'call', 'name': c['name'], 'args': c['args']}
                                tool_events.append(ev)
                                with tool_slot.expander(f"🔧 `{c['name']}` 호출", expanded=False):
                                    st.json(c['args'])
                        elif m.__class__.__name__ == 'ToolMessage':
                            ev = {
                                'kind': 'result',
                                'name': getattr(m, 'name', '(tool)'),
                                'content': str(m.content),
                            }
                            tool_events.append(ev)
                            with tool_slot.expander(f"👁 `{ev['name']}` 결과", expanded=False):
                                st.code(ev['content'][:2000])
                        else:
                            if getattr(m, 'content', None):
                                final_answer = m.content
                                answer_slot.markdown(final_answer)
        except Exception as e:
            st.error(f'에이전트 실행 오류: {type(e).__name__}: {e}')
            st.stop()

        st.session_state.messages.append({
            'role': 'assistant',
            'content': final_answer or '(응답 없음)',
            'tool_events': tool_events,
        })

Overwriting app.py


## 4. 실행 안내

앞 셀 실행으로 현재 폴더에 `app.py`가 생성되었다. **터미널**을 열고 다음을 수행:

```bash
cd day1/session3_capstone
streamlit run app.py
```

기본 포트는 `8501`이다. 브라우저가 자동으로 `http://localhost:8501` 을 연다.

### 동작 확인 체크리스트
- [ ] 좌측 사이드바에 `thread_id`(UUID)가 표시되는지
- [ ] "해킹 방지대책의 주요 통제항목을 규정에서 찾아줘" 입력 → `search_regulation` 호출 expander가 뜨는지
- [ ] 이어서 "그중 무선통신망 관련 항목만 더 알려줘" 입력 → **앞선 턴을 기억**하며 답변하는지 (MemorySaver 작동)
- [ ] "🆕 새 대화" 버튼 → `thread_id`가 바뀌고 메시지 영역이 초기화되는지
- [ ] "1000만원 연 3.5% 90일 이자?" → `calculate_interest` 호출로 결과 반환하는지

### 흔한 이슈
| 증상 | 원인 / 해결 |
|---|---|
| `Chroma 인덱스가 없습니다` 에러 | 2-4 노트북을 먼저 실행해 인덱스 생성 |
| `OPENAI_API_KEY` 관련 에러 | 프로젝트 루트 `.env` 확인 |
| DuckDuckGo 검색이 간헐적 실패 | 레이트 리밋. 잠시 후 재시도하거나 다른 질의 사용 |
| 도구 호출이 표시되지 않음 | `stream_mode='updates'` 이벤트 파싱 로직 / LangGraph 버전 확인 |

## 5. 확장 과제 (선택)

1. **폐쇄망 전환 모드**: `web_search` 도구를 제거하고 사내 문서 검색 API를 새 도구로 연결해 대체해보기. `SYSTEM_PROMPT`도 함께 수정 필요.
2. **감사 로그**: 매 턴의 `tool_events`를 JSONL 파일로 append 기록. 폐쇄망 금융 도메인의 추적 요건 대응 연습.
3. **Human-in-the-Loop**: 2-6 노트북의 `interrupt_before=['tools']` 패턴을 이 앱에 적용해, 민감 도구 호출 전 사용자 승인 버튼을 노출.
4. **가드레일**: `SYSTEM_PROMPT`의 규칙을 어긴 응답(예: 도구를 쓰지 않은 규정 답변)을 후처리로 걸러내는 간단한 검증 레이어 추가.
5. **LangFuse 셀프호스팅 연동**: 각 `agent.stream` 호출을 LangFuse 콜백으로 트레이싱해 운영 관측성 확보.

## 더 읽어보기
- [Streamlit — Build a basic LLM chat app](https://docs.streamlit.io/develop/tutorials/llms/build-conversational-apps)
- [LangGraph — `create_react_agent`](https://langchain-ai.github.io/langgraph/reference/prebuilt/#create_react_agent)
- [LangGraph — MemorySaver](https://langchain-ai.github.io/langgraph/how-tos/persistence/)